# LLM Provider Validation Lab

Notebook isolé pour valider techniquement des providers LLM avant intégration dans RetainFlow.

Objectif: tester l'API, les clés, le chat, le JSON structuré, le tool calling, le streaming, les erreurs, la latence et la compatibilité avec les besoins du `SupervisorAgent`.

Ce notebook ne modifie pas le code applicatif. Il utilise uniquement des variables d'environnement et n'affiche jamais les secrets.


## 1. Environment

Cette section charge `.env`, détecte les clés disponibles et vérifie les dépendances locales. Les clés API ne sont jamais imprimées.


In [ ]:
from __future__ import annotations

import json
import os
import statistics
import time
import traceback
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

try:
    import pandas as pd
except ImportError:
    pd = None

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if load_dotenv is not None:
    load_dotenv(PROJECT_ROOT / ".env", override=False)
    load_dotenv(PROJECT_ROOT / ".env.local", override=True)
else:
    print("python-dotenv missing: install project dependencies before running provider tests.")

REQUIRED_PACKAGES = {
    "python-dotenv": load_dotenv is not None,
    "pandas": pd is not None,
}

for package, available in REQUIRED_PACKAGES.items():
    print(f"{package:<18} {'FOUND' if available else 'MISSING'}")


In [ ]:
KEYS = {
    "Gemini API key": "GEMINI_API_KEY",
    "Google API key": "GOOGLE_API_KEY",
    "Groq API key": "GROQ_API_KEY",
    "OpenRouter API key": "OPENROUTER_API_KEY",
    "OpenAI API key": "OPENAI_API_KEY",
}

for label, env_name in KEYS.items():
    print(f"{label:<22} {'Found' if bool(os.getenv(env_name)) else 'Missing'}")


## 2. Configuration

Change `ACTIVE_PROVIDERS`, les modèles ou les paramètres globaux ici. Le notebook continue même si un provider n'est pas configuré.


In [ ]:
ACTIVE_PROVIDERS = ["gemini", "groq", "openrouter", "openai"]

DEFAULT_TIMEOUT_SECONDS = 20
DEFAULT_TEMPERATURE = 0.0
DEFAULT_MAX_TOKENS = 512
LATENCY_RUNS = 3

PROVIDERS = {
    "gemini": {
        "display": "Google Gemini",
        "type": "gemini_rest",
        "api_key_env": ["GEMINI_API_KEY", "GOOGLE_API_KEY"],
        "base_url": "https://generativelanguage.googleapis.com/v1beta",
        "model": os.getenv("GEMINI_MODEL", "gemini-1.5-flash"),
        "supports_chat": True,
        "supports_json": True,
        "supports_tools": True,
        "supports_streaming": True,
    },
    "groq": {
        "display": "Groq",
        "type": "openai_compatible",
        "api_key_env": ["GROQ_API_KEY"],
        "base_url": "https://api.groq.com/openai/v1",
        "model": os.getenv("GROQ_MODEL", os.getenv("LLM_MODEL", "llama-3.3-70b-versatile")),
        "supports_chat": True,
        "supports_json": True,
        "supports_tools": "model-dependent",
        "supports_streaming": True,
    },
    "openrouter": {
        "display": "OpenRouter",
        "type": "openai_compatible",
        "api_key_env": ["OPENROUTER_API_KEY"],
        "base_url": "https://openrouter.ai/api/v1",
        "model": os.getenv("OPENROUTER_MODEL", "openai/gpt-4o-mini"),
        "supports_chat": True,
        "supports_json": "model-dependent",
        "supports_tools": "model-dependent",
        "supports_streaming": True,
        "extra_headers": {
            "HTTP-Referer": os.getenv("OPENROUTER_SITE_URL", "http://localhost"),
            "X-Title": os.getenv("OPENROUTER_APP_NAME", "RetainFlow LLM Provider Validation Lab"),
        },
    },
    "openai": {
        "display": "OpenAI",
        "type": "openai_compatible",
        "api_key_env": ["OPENAI_API_KEY"],
        "base_url": "https://api.openai.com/v1",
        "model": os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        "supports_chat": True,
        "supports_json": True,
        "supports_tools": True,
        "supports_streaming": True,
    },
}


def first_env(names: list[str]) -> str | None:
    for name in names:
        value = os.getenv(name)
        if value:
            return value
    return None


def configured_providers() -> dict[str, dict[str, Any]]:
    selected = {}
    for name in ACTIVE_PROVIDERS:
        cfg = PROVIDERS[name].copy()
        cfg["api_key"] = first_env(cfg["api_key_env"])
        cfg["configured"] = bool(cfg["api_key"])
        selected[name] = cfg
    return selected

provider_configs = configured_providers()
for name, cfg in provider_configs.items():
    print(f"{cfg['display']:<16} model={cfg['model']:<32} configured={cfg['configured']}")


## 3. Provider Clients

Petite abstraction commune dans le notebook uniquement. Elle ne masque pas les capacités: chaque provider garde son implémentation native REST.


In [ ]:
@dataclass
class TestResult:
    provider: str
    model: str
    test: str
    status: str
    latency: float | None = None
    response: Any = None
    raw: Any = None
    error: dict[str, Any] | None = None
    metadata: dict[str, Any] = field(default_factory=dict)

RESULTS: list[TestResult] = []


def now() -> float:
    return time.perf_counter()


def post_json(url: str, payload: dict[str, Any], headers: dict[str, str], timeout: int = DEFAULT_TIMEOUT_SECONDS) -> dict[str, Any]:
    body = json.dumps(payload).encode("utf-8")
    req = Request(url, data=body, headers=headers, method="POST")
    with urlopen(req, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def stream_post(url: str, payload: dict[str, Any], headers: dict[str, str], timeout: int = DEFAULT_TIMEOUT_SECONDS):
    body = json.dumps(payload).encode("utf-8")
    req = Request(url, data=body, headers=headers, method="POST")
    return urlopen(req, timeout=timeout)


def classify_provider_error(error: BaseException) -> dict[str, Any]:
    message = str(error)
    status = None
    body = None
    if isinstance(error, HTTPError):
        status = error.code
        try:
            body = error.read().decode("utf-8")
            message = body or message
        except Exception:
            body = None
    elif isinstance(error, TimeoutError):
        return {"type": "timeout", "retryable": True, "http_status": None, "message": message}
    elif isinstance(error, URLError):
        return {"type": "connection_error", "retryable": True, "http_status": None, "message": message}

    lowered = message.lower()
    if status in {401, 403} or "api key" in lowered or "unauthorized" in lowered or "forbidden" in lowered:
        error_type, retryable = "authentication_error", False
    elif status == 404 or "model not found" in lowered or "does not exist" in lowered:
        error_type, retryable = "model_not_found", False
    elif status == 429 or "quota" in lowered or "rate limit" in lowered or "resource exhausted" in lowered or "too many requests" in lowered:
        error_type, retryable = "rate_limit", True
    elif status and 500 <= status < 600:
        error_type, retryable = "provider_unavailable", True
    elif status == 400 or "invalid" in lowered:
        error_type, retryable = "invalid_request", False
    else:
        error_type, retryable = "unknown_error", False
    return {"type": error_type, "retryable": retryable, "http_status": status, "message": message[:1200], "body": body}


def record(result: TestResult) -> TestResult:
    RESULTS.append(result)
    return result


def print_result(result: TestResult) -> None:
    print(f"Provider : {result.provider}")
    print(f"Model    : {result.model}")
    print(f"Test     : {result.test}")
    print(f"Status   : {result.status}")
    if result.latency is not None:
        print(f"Latency  : {result.latency:.2f} sec")
    if result.error:
        print("Error    :", json.dumps(result.error, indent=2, ensure_ascii=False))
    else:
        print("Response :", result.response)


In [ ]:
def openai_headers(cfg: dict[str, Any]) -> dict[str, str]:
    headers = {
        "Authorization": f"Bearer {cfg['api_key']}",
        "Content-Type": "application/json",
    }
    headers.update(cfg.get("extra_headers", {}))
    return headers


def openai_chat_payload(cfg: dict[str, Any], messages: list[dict[str, str]], *, temperature=DEFAULT_TEMPERATURE, max_tokens=DEFAULT_MAX_TOKENS, response_format=None, tools=None, stream=False) -> dict[str, Any]:
    payload = {
        "model": cfg["model"],
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": stream,
    }
    if response_format:
        payload["response_format"] = response_format
    if tools:
        payload["tools"] = tools
        payload["tool_choice"] = "auto"
    return payload


def gemini_url(cfg: dict[str, Any], method: str = "generateContent") -> str:
    return f"{cfg['base_url']}/models/{cfg['model']}:{method}?key={cfg['api_key']}"


def gemini_payload_from_messages(messages: list[dict[str, str]], *, temperature=DEFAULT_TEMPERATURE, max_tokens=DEFAULT_MAX_TOKENS, response_schema=None, tools=None) -> dict[str, Any]:
    system_instruction = None
    contents = []
    for message in messages:
        role = message["role"]
        text = message["content"]
        if role == "system":
            system_instruction = {"parts": [{"text": text}]}
        else:
            contents.append({"role": "model" if role == "assistant" else "user", "parts": [{"text": text}]})
    payload = {
        "contents": contents,
        "generationConfig": {
            "temperature": temperature,
            "maxOutputTokens": max_tokens,
        },
    }
    if system_instruction:
        payload["systemInstruction"] = system_instruction
    if response_schema:
        payload["generationConfig"]["responseMimeType"] = "application/json"
        payload["generationConfig"]["responseSchema"] = response_schema
    if tools:
        payload["tools"] = tools
    return payload


def extract_text(provider_type: str, raw: dict[str, Any]) -> str:
    if provider_type == "openai_compatible":
        return raw.get("choices", [{}])[0].get("message", {}).get("content", "") or ""
    if provider_type == "gemini_rest":
        parts = raw.get("candidates", [{}])[0].get("content", {}).get("parts", [])
        return "".join(str(part.get("text", "")) for part in parts)
    return ""


def extract_usage(provider_type: str, raw: dict[str, Any]) -> dict[str, Any]:
    if provider_type == "openai_compatible":
        usage = raw.get("usage") or {}
        return {
            "input_tokens": usage.get("prompt_tokens"),
            "output_tokens": usage.get("completion_tokens"),
            "total_tokens": usage.get("total_tokens"),
        }
    if provider_type == "gemini_rest":
        usage = raw.get("usageMetadata") or {}
        return {
            "input_tokens": usage.get("promptTokenCount"),
            "output_tokens": usage.get("candidatesTokenCount"),
            "total_tokens": usage.get("totalTokenCount"),
        }
    return {"input_tokens": None, "output_tokens": None, "total_tokens": None}


def call_llm(provider_name: str, messages: list[dict[str, str]], **kwargs) -> TestResult:
    cfg = provider_configs[provider_name]
    if not cfg["configured"]:
        return record(TestResult(cfg["display"], cfg["model"], kwargs.get("test", "chat"), "SKIPPED", error={"type": "missing_api_key", "message": "Provider not configured"}))
    started = now()
    try:
        if cfg["type"] == "openai_compatible":
            payload = openai_chat_payload(cfg, messages, temperature=kwargs.get("temperature", DEFAULT_TEMPERATURE), max_tokens=kwargs.get("max_tokens", DEFAULT_MAX_TOKENS), response_format=kwargs.get("response_format"), tools=kwargs.get("tools"), stream=False)
            raw = post_json(f"{cfg['base_url']}/chat/completions", payload, openai_headers(cfg), timeout=kwargs.get("timeout", DEFAULT_TIMEOUT_SECONDS))
        elif cfg["type"] == "gemini_rest":
            payload = gemini_payload_from_messages(messages, temperature=kwargs.get("temperature", DEFAULT_TEMPERATURE), max_tokens=kwargs.get("max_tokens", DEFAULT_MAX_TOKENS), response_schema=kwargs.get("response_schema"), tools=kwargs.get("gemini_tools"))
            raw = post_json(gemini_url(cfg), payload, {"Content-Type": "application/json"}, timeout=kwargs.get("timeout", DEFAULT_TIMEOUT_SECONDS))
        else:
            raise ValueError(f"Unsupported provider type: {cfg['type']}")
        latency = now() - started
        text = extract_text(cfg["type"], raw)
        return record(TestResult(cfg["display"], cfg["model"], kwargs.get("test", "chat"), "SUCCESS", latency, text, raw, metadata={"usage": extract_usage(cfg["type"], raw)}))
    except Exception as exc:
        latency = now() - started
        return record(TestResult(cfg["display"], cfg["model"], kwargs.get("test", "chat"), "FAILED", latency, error=classify_provider_error(exc)))


## 4. Connectivity

Prompt minimal: `Reply only with: API_OK`.


In [ ]:
for provider in ACTIVE_PROVIDERS:
    result = call_llm(provider, [{"role": "user", "content": "Reply only with: API_OK"}], test="connectivity", max_tokens=20)
    print_result(result)
    print("-" * 80)


## 5. Basic Chat

Vérifie que le message système est compris et compare la réponse brute avec une réponse normalisée.


In [ ]:
for provider in ACTIVE_PROVIDERS:
    result = call_llm(
        provider,
        [
            {"role": "system", "content": "You are a concise customer retention assistant."},
            {"role": "user", "content": "Explain churn in one sentence."},
        ],
        test="basic_chat",
        max_tokens=120,
    )
    print_result(result)
    if result.raw:
        print("Raw response keys:", list(result.raw.keys()))
        print("Normalized:", {"text": result.response, "usage": result.metadata.get("usage")})
    print("-" * 80)


## 6. French Instruction Following

Teste une réponse en français avec exactement trois raisons.


In [ ]:
FRENCH_PROMPT = "Réponds uniquement en français et donne exactement trois raisons qui peuvent expliquer le churn d'un client."

for provider in ACTIVE_PROVIDERS:
    result = call_llm(provider, [{"role": "user", "content": FRENCH_PROMPT}], test="french_instruction", max_tokens=180)
    text = result.response or ""
    likely_french = any(word in text.lower() for word in ["client", "prix", "service", "satisfaction", "résiliation", "fidel", "fidél"])
    approx_three = sum(text.count(marker) for marker in ["1", "2", "3", "-", "•"]) >= 3
    result.metadata.update({"likely_french": likely_french, "approx_three_items": approx_three})
    print_result(result)
    print("Checks:", result.metadata)
    print("-" * 80)


## 7. Structured Output / JSON

Teste une décision structurée proche du SupervisorAgent. Structured output natif est utilisé quand possible; sinon JSON prompting classique.


In [ ]:
ROUTING_SCHEMA = {
    "type": "object",
    "properties": {
        "intent": {"type": "string"},
        "customer_id": {"type": "string"},
        "needs_sql": {"type": "boolean"},
        "needs_model": {"type": "boolean"},
        "needs_shap": {"type": "boolean"},
        "needs_rag": {"type": "boolean"},
    },
    "required": ["intent", "customer_id", "needs_sql", "needs_model", "needs_shap", "needs_rag"],
}

json_instruction = "Analyse la question et retourne uniquement un JSON avec les champs: intent, customer_id, needs_sql, needs_model, needs_shap, needs_rag. Question: Pourquoi CUST_00123 risque de churn ?"

for provider in ACTIVE_PROVIDERS:
    cfg = provider_configs[provider]
    kwargs = {"test": "structured_output", "max_tokens": 220}
    if cfg["type"] == "openai_compatible":
        kwargs["response_format"] = {"type": "json_object"}
    elif cfg["type"] == "gemini_rest":
        kwargs["response_schema"] = ROUTING_SCHEMA
    result = call_llm(provider, [{"role": "user", "content": json_instruction}], **kwargs)
    parsed = None
    valid = False
    try:
        parsed = json.loads(result.response or "")
        valid = all(key in parsed for key in ROUTING_SCHEMA["required"])
    except Exception as exc:
        parsed = {"parse_error": str(exc)}
    result.metadata.update({"json_valid": valid, "parsed": parsed})
    print_result(result)
    print("Parsed:", json.dumps(parsed, indent=2, ensure_ascii=False))
    print("-" * 80)


## 8. Intent Routing

Compare les providers sur les questions typiques RetainFlow.


In [ ]:
ROUTING_QUESTIONS = [
    "Combien avons-nous de clients ?",
    "Montre-moi les 10 clients les plus à risque.",
    "Pourquoi CUST_00123 risque de churn ?",
    "Quelle stratégie de rétention recommandes-tu ?",
    "Rédige un email pour CUST_00123.",
    "Montre-moi le churn par région.",
]

routing_prompt = """
Tu es le routeur du SupervisorAgent RetainFlow. Retourne uniquement un JSON:
{
  "intent": "data_count|data_query|retention|customer_profile|strategy|email|visualization|kpi|unsupported",
  "required_capabilities": ["..."],
  "customer_ids": ["..."],
  "needs_visualization": false,
  "needs_rag": false,
  "needs_email": false
}
Question: {question}
"""

routing_rows = []
for provider in ACTIVE_PROVIDERS:
    for question in ROUTING_QUESTIONS:
        result = call_llm(provider, [{"role": "user", "content": routing_prompt.format(question=question)}], test="intent_routing", response_format={"type": "json_object"} if provider_configs[provider]["type"] == "openai_compatible" else None, max_tokens=260)
        parsed = None
        try:
            parsed = json.loads(result.response or "")
        except Exception:
            parsed = None
        routing_rows.append({"provider": result.provider, "model": result.model, "question": question, "status": result.status, "intent": (parsed or {}).get("intent"), "raw": result.response, "error": result.error})

if pd:
    display(pd.DataFrame(routing_rows))
else:
    print(json.dumps(routing_rows, indent=2, ensure_ascii=False))


## 9. Tool Calling

Teste le tool calling natif avec des outils mock. Aucun appel réel à la base de données.


In [ ]:
MOCK_TOOLS_OPENAI = [
    {"type": "function", "function": {"name": "get_customer", "description": "Get one customer by id.", "parameters": {"type": "object", "properties": {"customer_id": {"type": "string"}}, "required": ["customer_id"]}}},
    {"type": "function", "function": {"name": "get_top_churn_customers", "description": "Get top churn-risk customers.", "parameters": {"type": "object", "properties": {"limit": {"type": "integer"}}, "required": ["limit"]}}},
    {"type": "function", "function": {"name": "get_customer_count", "description": "Count customers in the database.", "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {"name": "search_retention_strategy", "description": "Search retention strategy documents.", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "draft_email", "description": "Draft a retention email.", "parameters": {"type": "object", "properties": {"customer_id": {"type": "string"}, "strategy": {"type": "string"}}, "required": ["customer_id"]}}},
]

MOCK_TOOLS_GEMINI = [
    {"functionDeclarations": [
        {"name": "get_customer", "description": "Get one customer by id.", "parameters": {"type": "object", "properties": {"customer_id": {"type": "string"}}, "required": ["customer_id"]}},
        {"name": "get_top_churn_customers", "description": "Get top churn-risk customers.", "parameters": {"type": "object", "properties": {"limit": {"type": "integer"}}, "required": ["limit"]}},
        {"name": "get_customer_count", "description": "Count customers in the database.", "parameters": {"type": "object", "properties": {}}},
        {"name": "search_retention_strategy", "description": "Search retention strategy documents.", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
        {"name": "draft_email", "description": "Draft a retention email.", "parameters": {"type": "object", "properties": {"customer_id": {"type": "string"}, "strategy": {"type": "string"}}, "required": ["customer_id"]}},
    ]}
]

TOOL_TESTS = [
    ("Combien avons-nous de clients ?", "get_customer_count"),
    ("Quelle stratégie recommandes-tu pour CUST_00123 ?", "get_customer + search_retention_strategy"),
    ("Rédige un email pour CUST_00123.", "get_customer + draft_email"),
]


def extract_tool_calls(provider_type: str, raw: dict[str, Any]) -> list[dict[str, Any]]:
    if provider_type == "openai_compatible":
        calls = raw.get("choices", [{}])[0].get("message", {}).get("tool_calls", []) or []
        return [{"name": call.get("function", {}).get("name"), "arguments": call.get("function", {}).get("arguments")} for call in calls]
    if provider_type == "gemini_rest":
        parts = raw.get("candidates", [{}])[0].get("content", {}).get("parts", [])
        return [{"name": part.get("functionCall", {}).get("name"), "arguments": part.get("functionCall", {}).get("args")} for part in parts if "functionCall" in part]
    return []

for provider in ACTIVE_PROVIDERS:
    cfg = provider_configs[provider]
    for question, expected in TOOL_TESTS:
        kwargs = {"test": "tool_calling", "max_tokens": 180}
        if cfg["type"] == "openai_compatible":
            kwargs["tools"] = MOCK_TOOLS_OPENAI
        elif cfg["type"] == "gemini_rest":
            kwargs["gemini_tools"] = MOCK_TOOLS_GEMINI
        result = call_llm(provider, [{"role": "user", "content": question}], **kwargs)
        tool_calls = extract_tool_calls(cfg["type"], result.raw or {}) if result.raw else []
        support = "SUPPORTED" if tool_calls else ("FAILED" if result.status == "FAILED" else "PARTIAL/NOT SUPPORTED")
        print(f"Provider={result.provider} | Question={question}")
        print("Expected:", expected)
        print("Support :", support)
        print("Tool requested:", tool_calls)
        print("Raw provider response:", json.dumps(result.raw, indent=2, ensure_ascii=False)[:3000] if result.raw else result.error)
        print("-" * 80)


## 10. Multi-Step Tool Use

Simulation contrôlée avec tools mock. On montre un plan structuré, les tool calls, les outputs et la synthèse, sans exposer de chain-of-thought privée.


In [ ]:
MOCK_TOOL_OUTPUTS = {
    "get_top_churn_customers": {"customers": ["CUST_001", "CUST_002", "CUST_003"]},
    "get_customer": {"customer_id": "CUST_001", "segment": "PRICE_SENSITIVE", "churn_probability": 0.64},
    "search_retention_strategy": {"strategy": "Offer controlled loyalty review and advisor call."},
    "draft_email": {"subject": "Votre contrat", "body": "Bonjour, faisons le point ensemble."},
}

MULTI_STEP_PROMPT = """
Tu es un superviseur RetainFlow. Pour la demande ci-dessous, retourne uniquement un JSON avec:
- plan: liste courte d'étapes publiques
- tool_calls: liste de tools à appeler avec arguments
- final_synthesis: synthèse courte basée sur les sorties mock disponibles
Demande: Trouve les 3 clients les plus à risque puis propose une stratégie de rétention pour chacun.
Tools disponibles: get_top_churn_customers, get_customer, search_retention_strategy, draft_email.
Sorties mock disponibles: {mock_outputs}
"""

for provider in ACTIVE_PROVIDERS:
    result = call_llm(provider, [{"role": "user", "content": MULTI_STEP_PROMPT.format(mock_outputs=json.dumps(MOCK_TOOL_OUTPUTS, ensure_ascii=False))}], test="multi_step_tools", response_format={"type": "json_object"} if provider_configs[provider]["type"] == "openai_compatible" else None, max_tokens=700)
    print_result(result)
    try:
        print(json.dumps(json.loads(result.response or "{}"), indent=2, ensure_ascii=False))
    except Exception:
        pass
    print("-" * 80)


## 11. Streaming

Mesure le time-to-first-token et la latence totale quand le provider supporte le streaming.


In [ ]:
def stream_llm(provider_name: str, prompt: str) -> TestResult:
    cfg = provider_configs[provider_name]
    if not cfg["configured"]:
        return record(TestResult(cfg["display"], cfg["model"], "streaming", "SKIPPED", error={"type": "missing_api_key", "message": "Provider not configured"}))
    started = now()
    first_token_at = None
    chunks = []
    try:
        if cfg["type"] == "openai_compatible":
            payload = openai_chat_payload(cfg, [{"role": "user", "content": prompt}], max_tokens=120, stream=True)
            with stream_post(f"{cfg['base_url']}/chat/completions", payload, openai_headers(cfg)) as response:
                for raw_line in response:
                    line = raw_line.decode("utf-8", errors="ignore").strip()
                    if not line.startswith("data:") or line == "data: [DONE]":
                        continue
                    if first_token_at is None:
                        first_token_at = now()
                    chunks.append(line[5:].strip())
        elif cfg["type"] == "gemini_rest":
            payload = gemini_payload_from_messages([{"role": "user", "content": prompt}], max_tokens=120)
            with stream_post(gemini_url(cfg, method="streamGenerateContent"), payload, {"Content-Type": "application/json"}) as response:
                body = response.read().decode("utf-8", errors="ignore")
                first_token_at = now()
                chunks.append(body[:3000])
        total = now() - started
        return record(TestResult(cfg["display"], cfg["model"], "streaming", "SUCCESS", total, "".join(chunks)[:1000], metadata={"time_to_first_token": None if first_token_at is None else first_token_at - started, "streaming_supported": True}))
    except Exception as exc:
        total = now() - started
        return record(TestResult(cfg["display"], cfg["model"], "streaming", "FAILED", total, error=classify_provider_error(exc), metadata={"streaming_supported": False}))

for provider in ACTIVE_PROVIDERS:
    result = stream_llm(provider, "Reply in one short sentence about retention analytics.")
    print_result(result)
    print("Metadata:", result.metadata)
    print("-" * 80)


## 12. Error Handling

Tests contrôlés: modèle invalide, clé absente simulée, timeout court. Pas de spam ni test de quota agressif.


In [ ]:
ERROR_TESTS = []

for provider in ACTIVE_PROVIDERS:
    cfg = provider_configs[provider]
    if not cfg["configured"]:
        continue

    original = cfg["model"]
    cfg["model"] = "definitely-not-a-real-model-xyz"
    result = call_llm(provider, [{"role": "user", "content": "Hello"}], test="error_invalid_model", max_tokens=10)
    ERROR_TESTS.append(result)
    cfg["model"] = original

    saved_key = cfg["api_key"]
    cfg["api_key"] = "invalid-key-for-test"
    result = call_llm(provider, [{"role": "user", "content": "Hello"}], test="error_invalid_key", max_tokens=10)
    ERROR_TESTS.append(result)
    cfg["api_key"] = saved_key

    result = call_llm(provider, [{"role": "user", "content": "Hello"}], test="error_timeout", max_tokens=10, timeout=0.001)
    ERROR_TESTS.append(result)

for result in ERROR_TESTS:
    print_result(result)
    print("Classified as:", result.error)
    print("-" * 80)


## 13. Latency Benchmark

Trois appels maximum par provider configuré. Benchmark raisonnable, non agressif.


In [ ]:
latency_rows = []
for provider in ACTIVE_PROVIDERS:
    latencies = []
    statuses = []
    for idx in range(LATENCY_RUNS):
        result = call_llm(provider, [{"role": "user", "content": "Reply with one word: OK"}], test="latency", max_tokens=10)
        statuses.append(result.status)
        if result.latency is not None and result.status == "SUCCESS":
            latencies.append(result.latency)
    cfg = provider_configs[provider]
    latency_rows.append({
        "provider": cfg["display"],
        "model": cfg["model"],
        "successes": statuses.count("SUCCESS"),
        "mean_latency": round(statistics.mean(latencies), 3) if latencies else None,
        "min_latency": round(min(latencies), 3) if latencies else None,
        "max_latency": round(max(latencies), 3) if latencies else None,
    })

if pd:
    display(pd.DataFrame(latency_rows))
else:
    print(json.dumps(latency_rows, indent=2, ensure_ascii=False))


## 14. Token / Usage Information

Affiche les tokens seulement si le provider les retourne. Aucune valeur n'est inventée.


In [ ]:
usage_rows = []
for result in RESULTS:
    if result.status == "SUCCESS":
        usage = result.metadata.get("usage") or {}
        if usage:
            usage_rows.append({"provider": result.provider, "model": result.model, "test": result.test, **usage})

if pd and usage_rows:
    display(pd.DataFrame(usage_rows).drop_duplicates())
else:
    print(json.dumps(usage_rows or [{"message": "No usage information returned yet."}], indent=2, ensure_ascii=False))


## 15. Project-Specific Supervisor Tests

Teste si le modèle peut devenir cerveau du `SupervisorAgent` RetainFlow. Capacités réelles repérées dans le repo: SQL/KPI, Customer Profile, Churn prediction, Prioritization, Explainability, RAG Strategy, Visualization, Email generation.


In [ ]:
SUPERVISOR_TESTS = [
    "Combien avons-nous de clients ?",
    "Donne-moi les 10 clients les plus susceptibles de churn.",
    "Pourquoi CUST_00123 est à risque ?",
    "Quelle stratégie recommandes-tu pour CUST_00123 ?",
    "Rédige un email de rétention pour ce client.",
    "Montre-moi le churn par région.",
    "Trouve les 5 clients les plus à risque, propose une stratégie et prépare un email pour chacun.",
]

SUPERVISOR_PROMPT = """
Tu es le cerveau candidat du SupervisorAgent RetainFlow. Décide des capacités nécessaires.
Retourne uniquement un JSON:
{
  "intent": "data_count|data_query|retention|customer_profile|strategy|email|visualization|kpi|multi_step|unsupported",
  "required_capabilities": ["SQL/KPI", "Customer Profile", "Churn prediction", "Prioritization", "Explainability", "RAG Strategy", "Visualization", "Email generation"],
  "customer_ids": [],
  "needs_sql": false,
  "needs_model": false,
  "needs_shap": false,
  "needs_rag": false,
  "needs_visualization": false,
  "needs_email": false,
  "should_answer_or_refuse": "answer|refuse",
  "reason": "courte justification"
}
Question: {question}
"""

supervisor_rows = []
for provider in ACTIVE_PROVIDERS:
    for question in SUPERVISOR_TESTS:
        result = call_llm(provider, [{"role": "user", "content": SUPERVISOR_PROMPT.format(question=question)}], test="supervisor_specific", response_format={"type": "json_object"} if provider_configs[provider]["type"] == "openai_compatible" else None, max_tokens=500)
        parsed = {}
        try:
            parsed = json.loads(result.response or "{}")
        except Exception:
            parsed = {"parse_error": result.response}
        supervisor_rows.append({"provider": result.provider, "model": result.model, "question": question, "status": result.status, "intent": parsed.get("intent"), "capabilities": parsed.get("required_capabilities"), "raw": parsed, "error": result.error})

if pd:
    display(pd.DataFrame(supervisor_rows))
else:
    print(json.dumps(supervisor_rows, indent=2, ensure_ascii=False))


## 16. Capability Matrix

Matrice générée automatiquement à partir des résultats collectés.


In [ ]:
def mark(condition: bool | None) -> str:
    if condition is True:
        return "OK"
    if condition is False:
        return "NO"
    return "N/A"

matrix_rows = []
for provider in ACTIVE_PROVIDERS:
    cfg = provider_configs[provider]
    provider_results = [r for r in RESULTS if r.provider == cfg["display"]]
    def ok(test_name: str):
        matches = [r for r in provider_results if r.test == test_name]
        if not matches:
            return None
        return any(r.status == "SUCCESS" for r in matches)
    latencies = [r.latency for r in provider_results if r.status == "SUCCESS" and r.latency is not None]
    matrix_rows.append({
        "Provider": cfg["display"],
        "Model": cfg["model"],
        "Chat": mark(ok("basic_chat")),
        "JSON": mark(any((r.metadata.get("json_valid") is True) for r in provider_results if r.test == "structured_output")),
        "Tool Calling": mark(any(extract_tool_calls(cfg["type"], r.raw or {}) for r in provider_results if r.test == "tool_calling")),
        "Streaming": mark(ok("streaming")),
        "Latency": round(statistics.mean(latencies), 2) if latencies else "N/A",
        "Status": "READY" if ok("connectivity") else ("NOT CONFIGURED" if not cfg["configured"] else "CHECK"),
    })

capability_matrix = pd.DataFrame(matrix_rows) if pd else matrix_rows
if pd:
    display(capability_matrix)
else:
    print(json.dumps(capability_matrix, indent=2, ensure_ascii=False))


## 17. Scoring

Score technique simple sur 100. Ce n'est pas un benchmark scientifique; c'est une aide à la décision pour RetainFlow.


In [ ]:
WEIGHTS = {
    "connectivity": 10,
    "french_instruction": 10,
    "structured_output": 15,
    "tool_calling": 20,
    "multi_step_tools": 15,
    "basic_chat": 10,
    "latency": 5,
    "error_handling": 5,
    "streaming": 5,
    "supervisor_specific": 5,
}

score_rows = []
for provider in ACTIVE_PROVIDERS:
    cfg = provider_configs[provider]
    provider_results = [r for r in RESULTS if r.provider == cfg["display"]]
    score = 0
    details = {}
    for test_name, weight in WEIGHTS.items():
        if test_name == "error_handling":
            passed = any(r.test.startswith("error_") and r.error for r in provider_results)
        elif test_name == "latency":
            passed = any(r.test == "latency" and r.status == "SUCCESS" and (r.latency or 999) < 5 for r in provider_results)
        elif test_name == "tool_calling":
            passed = any(extract_tool_calls(cfg["type"], r.raw or {}) for r in provider_results if r.test == "tool_calling")
        elif test_name == "structured_output":
            passed = any(r.test == "structured_output" and r.metadata.get("json_valid") for r in provider_results)
        else:
            passed = any(r.test == test_name and r.status == "SUCCESS" for r in provider_results)
        details[test_name] = weight if passed else 0
        score += details[test_name]
    score_rows.append({"provider": cfg["display"], "model": cfg["model"], "score": score, **details})

if pd:
    display(pd.DataFrame(score_rows).sort_values("score", ascending=False))
else:
    print(json.dumps(score_rows, indent=2, ensure_ascii=False))


## 18. Final Recommendation

Synthèse automatique. À lire avant toute intégration dans `SupervisorAgent`.


In [ ]:
recommendations = []
for row in score_rows:
    score = row["score"]
    if score >= 80:
        status = "YES"
    elif score >= 60:
        status = "WITH LIMITATIONS"
    else:
        status = "NO"
    recommendations.append({
        "Provider": row["provider"],
        "Model": row["model"],
        "Overall score": score,
        "Recommended for Supervisor": status,
        "Recommended for routing": "YES" if row.get("structured_output", 0) and row.get("basic_chat", 0) else "NO",
        "Recommended for synthesis": "YES" if row.get("basic_chat", 0) and row.get("french_instruction", 0) else "WITH LIMITATIONS",
        "Recommended for email drafting": "YES" if row.get("french_instruction", 0) else "NO",
        "Recommended for RAG answer generation": "YES" if row.get("structured_output", 0) and row.get("basic_chat", 0) else "WITH LIMITATIONS",
        "Recommended for fallback": "YES" if score >= 70 else "NO",
        "Main limitations": "Check failed tests, quota/rate limits, tool calling support and JSON reliability.",
    })

if pd:
    display(pd.DataFrame(recommendations))
else:
    print(json.dumps(recommendations, indent=2, ensure_ascii=False))


## 19. Integration Recommendations

Notes pour RetainFlow, sans modification du code métier depuis ce notebook:

- Garder les tools déterministes (`SQLTool`, `CustomerProfileTool`, `ExplainabilityTool`, `StrategyRAGTool`) comme garde-fous.
- Utiliser le LLM pour router, rédiger et synthétiser uniquement à partir de preuves contrôlées.
- Refuser une demande non comprise au lieu de déclencher une route par défaut.
- Activer la rédaction LLM client uniquement avec consentement explicite car cela peut envoyer des preuves client au provider.
- Avant intégration d'un nouveau modèle, exécuter ce lab et vérifier JSON, tool calling, streaming, latence et erreurs.
